# Quantization Lab: one model, FP16 vs every quant (Kaggle, 2× T4)

**Repo:** https://github.com/jprakash-1/model-quantization-exp · **Model:** `Qwen/Qwen3-4B-Instruct-2507` (4.02B params, non-thinking, greedy decoding)

Each quantized variant is compared to the 16-bit original on:

| | Metric | How |
|---|---|---|
| Memory | weights size, bits per weight, **peak GPU memory** | `get_memory_footprint()` / file size, NVML sampling |
| Speed | prefill tok/s, **decode tok/s**, TTFT for a 512-token prompt | single stream, median of 3 |
| Drift | **KL divergence** vs baseline, top-1 token agreement, perplexity | wikitext-2, 20 × 512 tokens |
| Quality | **GSM8K** strict/flexible, **ARC-Challenge**, answers lost/gained vs baseline | 250 questions each |

### The hardware, and what it means
`2 × Tesla T4`: 15 GB each, Turing (compute capability 7.5), ~320 GB/s, driver 580 / CUDA 13.0.
- **Both tracks run at once, one per GPU**, so a full run fits in one 12 h session:

  | GPU 0 (this notebook's kernel) | GPU 1 (a background job) |
  |---|---|
  | **Track 1, Hugging Face Transformers:** bitsandbytes INT8 / NF4 / FP4, Unsloth dynamic 4-bit, HQQ 4/3/2-bit, FP8 | **Track 2, llama.cpp GGUF:** Q8_0 down to Q2_K, ± Unsloth's imatrix, Unsloth Dynamic `UD-*` |
- The model in FP16 is 8 GB, so it fits on one T4 with room for batched generation. Never split a model across both GPUs here.
- **The baseline is FP16**, because Turing has no BF16. **FP8 needs compute capability ≥ 8.9, so it will fail**. That's a result: your GPU decides the format.
- Each track is compared only against its own baseline. Compare speed within a track, not across tracks.

### Kaggle settings (right sidebar)
- **Accelerator: GPU T4 ×2**, **Internet: On**
- First time: set `N_QUESTIONS = 100` and run cell by cell. Full run: set 250 and use **Save Version → Save & Run All (Commit)**.

Every variant writes `results/<track>-<variant>/metrics.json`. **Re-running skips finished variants**, so a timeout costs nothing. A variant that fails writes `error.txt` and the run continues.

In [ ]:
# 1. Get the latest code + install
REPO_URL = "https://github.com/jprakash-1/model-quantization-exp.git"
REPO_DIR = "/kaggle/working/model-quantization-exp"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1
!pip install -q -r requirements-kaggle.txt   # no -U: keep Kaggle's preinstalled packages as they are
!pip list 2>/dev/null | grep -iE "^(torch|transformers|accelerate|bitsandbytes|hqq|datasets|openai|pandas) "
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# 2. Configure. This kernel only sees GPU 0 (set before torch is imported); GPU 1 is for the GGUF job.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from qlab import config
N_QUESTIONS = 250                              # 100 for a quick first pass
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"       # "Qwen/Qwen3-1.7B" for a ~2.5x faster run
config.N_QUESTIONS = N_QUESTIONS
config.set_model(MODEL_ID)

HF_VARIANTS = ["fp16", "bnb-int8", "bnb-nf4", "bnb-nf4-dq", "bnb-fp4", "unsloth-dyn4", "hqq4", "hqq3", "hqq2", "fp8"]
GGUF_VARIANTS = ["F16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_M-imat", "UD-Q4_K_XL",
                 "IQ4_XS-imat", "Q3_K_M-imat", "Q2_K", "Q2_K-imat", "UD-Q2_K_XL"]
print(config.MODEL_ID, config.N_QUESTIONS)
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# 3. Data: wikitext-2 (drift), GSM8K + ARC-Challenge (tasks) -> data/
from qlab import data
data.download(n=config.N_QUESTIONS)
!ls -la data

## Build llama.cpp, then download every model up front

1. **Build llama.cpp with CUDA for the T4 (~10–15 min).** The compile uses all 4 CPUs, so it must finish before any speed is measured.
2. **Download every checkpoint now (~30 GB, a few minutes on Kaggle).** That way `load_s` measures loading, not downloading, and a missing file shows up now instead of hours in. Variants already measured are skipped on a re-run.

| Track | Downloaded | Used by |
|---|---|---|
| HF | `Qwen/Qwen3-4B-Instruct-2507` (FP16, 8 GB) | `fp16`, all `bnb-*`, all `hqq*`, `torchao-int8` (quantized on load) |
| HF | `unsloth/…-unsloth-bnb-4bit` | `unsloth-dyn4` |
| HF | `Qwen/…-FP8` | `fp8` |
| GGUF | `unsloth/…-GGUF`: `F16.gguf` + `imatrix_unsloth.gguf_file` + `UD-Q4_K_XL` + `UD-Q2_K_XL` | baseline, the source for every local quant, `*-imat`, `UD-*` |

The other GGUF quants (`Q8_0` … `Q2_K`, `IQ4_XS`) aren't downloaded. `llama-quantize` makes them from the F16 file during the GGUF track, and deletes each one once it's measured.

In [ ]:
from qlab import gguf_backend as gg
gg.build()

In [ ]:
# 4. Download all models (HF cache in ~/.cache/huggingface, GGUF files in /tmp/qlab/gguf)
from qlab import hf_backend as hf
hf.download(HF_VARIANTS)
gg.prepare(GGUF_VARIANTS)
!du -sh ~/.cache/huggingface/hub /tmp/qlab/gguf; df -h /tmp | tail -1

## Track 2: llama.cpp / GGUF, in the background on GPU 1

Runs as a **background job on GPU 1**. It makes each quant from the F16 GGUF, measures it, deletes it, and logs to `results/gguf_track.log`.

| Variant | Lesson |
|---|---|
| `F16` | reference (Unsloth publishes F16 for this model, so it's also the quant source) |
| `Q8_0`, `Q6_K`, `Q5_K_M` | the near-lossless zone |
| `Q4_K_M` vs `Q4_K_M-imat` vs `UD-Q4_K_XL` | what calibration, then per-layer mixing, buy at 4-bit |
| `IQ4_XS-imat` | non-uniform 4-bit code book |
| `Q3_K_M-imat` | the 3-bit cliff |
| `Q2_K` vs `Q2_K-imat` vs `UD-Q2_K_XL` | can calibration and mixing rescue 2-bit? |

In [ ]:
# 5. Start the GGUF track on GPU 1
import subprocess, sys
GGUF_LOG = config.RESULTS / "gguf_track.log"
config.RESULTS.mkdir(exist_ok=True)

if "gguf_job" in globals() and gguf_job.poll() is None:
    print("GGUF track already running, pid", gguf_job.pid)
else:
    gguf_job = subprocess.Popen(
        [sys.executable, "-u", "-m", "qlab.run", "gguf", *GGUF_VARIANTS, "--n", str(N_QUESTIONS), "--model", MODEL_ID],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}, stdout=open(GGUF_LOG, "a"), stderr=subprocess.STDOUT)
    print("GGUF track started on GPU 1, pid", gguf_job.pid, "->", GGUF_LOG)
    print("Peek at it any time from a new cell:  !tail -n 5 {GGUF_LOG}")

## Track 1: Hugging Face Transformers, in this kernel on GPU 0

The **baseline runs first**: it saves the reference log-probs that every other variant's KL divergence is measured against.

| Variant | What it is | Lesson |
|---|---|---|
| `fp16` | original weights in FP16 (a T4 has no BF16) | reference |
| `bnb-int8` | LLM.int8(): INT8 weights, outlier features kept in 16-bit | "8-bit is free", but is it fast? |
| `bnb-nf4` | QLoRA's 4-bit NormalFloat, block size 64 | standard 4-bit |
| `bnb-nf4-dq` | NF4 plus quantized quantization scales | how much do the scales cost? |
| `bnb-fp4` | 4-bit float instead of NF4 | does the 4-bit *code book* matter? |
| `unsloth-dyn4` | Unsloth's NF4 with sensitive layers left in 16-bit | mixed precision vs uniform NF4 |
| `hqq4/3/2` | Half-Quadratic Quantization, no calibration data, group 64 | where is the cliff? |
| `fp8` | Qwen's official FP8 checkpoint *(needs compute capability ≥ 8.9; the T4 is 7.5)* | your GPU decides the format |
| `torchao-int8` | int8 weight-only *(optional cell below)* | |

In [ ]:
# 6. HF track on GPU 0 (this kernel)
from qlab import run
from qlab.hf_backend import BASELINE
assert HF_VARIANTS[0] == BASELINE, f"baseline on this GPU is {BASELINE}: it must run first"
run.run_all("hf", HF_VARIANTS)
print("
GPU 1 meanwhile:"); !tail -n 3 {GGUF_LOG}

In [ ]:
# Optional extra: needs one more library, which must match the installed torch version.
# !pip install -q torchao && python -c "import torchao"
# run.run("hf", "torchao-int8")

In [ ]:
# Wait for the GGUF track on GPU 1: every minute, show how many variants are done and its latest line
import time
while gguf_job.poll() is None:
    done = sum((config.RESULTS / f"gguf-{v}" / "metrics.json").exists() for v in GGUF_VARIANTS)
    last = (open(GGUF_LOG).read().strip().splitlines() or ["(no output yet)"])[-1].strip()[:140]
    print(f"{time.strftime('%H:%M')}  GGUF {done}/{len(GGUF_VARIANTS)} variants done | {last}", flush=True)
    time.sleep(60)
print("GGUF track exit code:", gguf_job.returncode)
!grep -E "^===|GSM8K [0-9.]+/|FAILED|took|TRACK DONE" {GGUF_LOG}

## Report

In [ ]:
from qlab import report
import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 30)

df, errors = report.table()
VIEWS = {
    "Memory": ["bpw", "size_gb", "peak_gpu_gb", "torch_peak_allocated_gb", "weights_vram_gb", "kv_cache_gb", "compute_buffer_gb"],
    "Speed & latency": ["prefill_tps", "ttft_ms", "decode_tps", "decode_tps_std", "gsm8k_throughput_tps",
                        "gsm8k_latency_p50_s", "gsm8k_latency_p95_s", "server_ttft_ms_p50", "server_decode_tps_mean"],
    "Quality": ["ppl", "mean_kld", "p99_kld", "top1_agree", "gsm8k_strict", "gsm8k_flexible", "strict_gap", "arc",
                "lost_vs_base", "gained_vs_base", "gsm8k_hit_max_tokens"],
    "Time (s)": ["quant_s", "load_s", "time_make_s", "time_load_s", "time_bench_s", "time_tasks_s", "time_kld_s", "time_total_s"],
}
for title, cols in VIEWS.items():
    print(f"
{title}")
    display(df[["track", "variant", *cols]].dropna(axis=1, how="all"))
print("Everything per variant: results/<track>-<variant>/metrics.json · all columns: results/summary.csv")
for name, err in errors.items():
    print(f"FAILED {name}: {err}")
report.plots(df);

In [ ]:
# Look at answers a variant got wrong that its baseline got right
import json
def lost(track, variant, k=3):
    base = {"hf": BASELINE, "gguf": "F16"}[track]
    b = json.load(open(config.RESULTS / f"{track}-{base}" / "eval.json"))["gsm8k"]
    q = json.load(open(config.RESULTS / f"{track}-{variant}" / "eval.json"))["gsm8k"]
    for x, y in [(x, y) for x, y in zip(b, q) if x["flexible"] and not y["flexible"]][:k]:
        print(f"--- question {x['i']}\nBASELINE: ...{x['reply'][-200:]}\n{variant.upper()}: ...{y['reply'][-200:]}\n")

lost("hf", "hqq3")

## Questions to answer from the table

1. Is 8-bit (`bnb-int8`, `Q8_0`) indistinguishable from the baseline? Look at mean KLD, net flips, and whether `bnb-int8` is actually *slower* than FP16.
2. At 4-bit, what matters more: the code book (NF4 vs FP4), the scales (`-dq`), calibration (`-imat`), or per-layer mixing (`unsloth-dyn4`, `UD-*`)?
3. Does perplexity move under 1% while KLD and GSM8K clearly move? (Ledger: "perplexity hides most of the damage")
4. Does GSM8K drop before ARC does? (Ledger: "the benchmark decides what you see")
5. Is *strict minus flexible* bigger for some quants, meaning they break the answer format before the math?
6. Where is the cliff for a 4B model: 3-bit or 2-bit? Does imatrix or UD move it?
7. Decode is memory-bound (~320 GB/s on a T4): does tok/s follow weight size? Which formats are smaller but *not* faster (dequantization overhead)?
8. Which variants use more GPU memory than their bits suggest, and why?
9. Your pick: which single variant would you deploy on one 15 GB T4, and why?

In [ ]:
# Wrap up: free the big files (weights, llama.cpp build, KLD refs). results/ stays in the notebook output.
run.cleanup()
!du -sh results && ls results

In [ ]:
# Optional: push results back to the repo. Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN first.
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git -C {REPO_DIR} config user.email "you@example.com" && git -C {REPO_DIR} config user.name "kaggle-run"
# !git -C {REPO_DIR} add results && git -C {REPO_DIR} commit -qm "Kaggle run results" && git -C {REPO_DIR} push -q https://{token}@{REPO_URL.removeprefix("https://")} HEAD